#Microservice-Identification-from-Runtime-Traces-using-LLMs
---

## What this Notebook does

1. You upload your trace log file
2. The Notebook builds **three matrices** from the traces:
   - **HTTP Route → Class matrix** — which classes handle which routes, with trace-count weights
   - **Class co-occurrence matrix** — how many distinct traces each pair of classes appeared in together
   - **Class × Table interaction matrix** — how many traces each class performed each DB operation on each table
3. It builds **one structured prompt** containing all three matrices
4. It runs the prompt on multiple LLMs
5. It perform Hierarchical Clustering and Louvain Algorithm on the Combined matrices
6. Download proposed microservice decomposition

---


In [ ]:
# Install all required libraries
!pip install -q pandas networkx matplotlib requests tabulate

import re, json, os, itertools, time
from collections import defaultdict
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import requests

print('All libraries ready.')

All libraries ready.


---
## Enter your API keys


In [ ]:

# API Keys ──────────────────────────────────────────────────────────────────
GROQ_API_KEY = ""
OPENROUTER_API_KEY = ""
OPENAI_API_KEY = ""
ANTHROPIC_API_KEY = "



ENDPOINTS = {
    "groq"      : "https://api.groq.com/openai/v1/chat/completions",
    "openrouter": "https://openrouter.ai/api/v1/chat/completions",
    "openai"    : "https://api.openai.com/v1/chat/completions",
    "anthropic" : "https://api.anthropic.com/v1/messages",
}

PROVIDER_KEYS = {
    "groq"      : GROQ_API_KEY,
    "openrouter": OPENROUTER_API_KEY,
    "openai"    : OPENAI_API_KEY,
    "anthropic" : ANTHROPIC_API_KEY,
}

MODELS = {
    "Llama 4 Scout"     : ("groq",       "meta-llama/llama-4-scout-17b-16e-instruct"),

    "DeepSeek V4 Flash" : ("openrouter", "deepseek/deepseek-v4-flash"),

    "Qwen3"        : ("openrouter", "qwen/qwen3-coder"),
    "GPT-5.4 Mini" : ("openai",     "gpt-5.4-mini"),

    "Claude Opus 4.8" : ("anthropic", "claude-opus-4-8"),

}

MAX_TOKENS  = 8192
TEMPERATURE = 0.0   # fully deterministic
N_RUNS      = 3     # number of runs per model

print('Configuration ready.')
print(f'Each model will run {N_RUNS}x for consistency measurement.')
print('Models that will be compared:')
for label, (provider, model_id) in MODELS.items():
    print(f'  {label:20s} → {model_id:50s} [{provider}]')


---
##  Upload your trace log file


In [ ]:
from google.colab import files as _colab_files

print('Select your OTEL trace log file.')
print('Waiting for upload...')

_uploaded = _colab_files.upload()

if not _uploaded:
    raise RuntimeError('No file uploaded. Re-run this cell and select your log file.')

RAW_LOG_LINES = []
LOG_FILENAME  = ''
for _fname, _fbytes in _uploaded.items():
    LOG_FILENAME = _fname
    try:
        _text = _fbytes.decode('utf-8')
    except UnicodeDecodeError:
        _text = _fbytes.decode('latin-1')
    RAW_LOG_LINES = [l.rstrip() for l in _text.splitlines() if l.strip()]
    break  # use first file only

print(f'Loaded: {LOG_FILENAME}  ({len(RAW_LOG_LINES):,} lines)')

if not RAW_LOG_LINES:
    raise RuntimeError('The uploaded file is empty.')

print('First 3 lines:')
for l in RAW_LOG_LINES[:3]:
    print(' ', l[:120])

---
## Parse the trace log and remove noise


In [ ]:
import re


TRACE_ID_RE = re.compile(r"'\s*:\s*([0-9a-fA-F]{32})\s+[0-9a-fA-F]{16}")
KIND_RE     = re.compile(r'\b(SERVER|CLIENT|INTERNAL|PRODUCER|CONSUMER)\b')
TRACER_RE   = re.compile(r'\[tracer:\s*([\w\-\.\:]+)')
DB_TABLE_RE = re.compile(r"'(?:SELECT|INSERT|UPDATE|DELETE)[^']{0,300}?\.(\w+)'", re.IGNORECASE)

ATTR_PATTERNS = {
    'code.namespace':            re.compile(r'code\.namespace=([^,}]+)'),
    'code.function':              re.compile(r'code\.function=([^,}]+)'),
    'http.route':                re.compile(r'http\.route=([^,}]+)'),
    'db.statement':               re.compile(r'db\.statement=([^,}]+)'),
    'db.sql.table':               re.compile(r'db\.sql\.table=([^,}]+)'),
    'url.path':                   re.compile(r'url\.path=([^,}]+)'),
    'url.query':                  re.compile(r'url\.query=([^,}]+)'),
    'http.response.status_code':  re.compile(r'http\.response\.status_code=([^,}]+)'),
}
def get_attr_compiled(line, pattern):
    m = pattern.search(line)
    return m.group(1).strip() if m else ''
def parse_line(line):
    span = {}

    m = TRACE_ID_RE.search(line)
    span['trace_id'] = m.group(1) if m else ''

    m = KIND_RE.search(line)
    span['kind'] = m.group(1) if m else ''

    m = TRACER_RE.search(line)
    span['tracer'] = m.group(1) if m else ''

    span['namespace']    = get_attr_compiled(line, ATTR_PATTERNS['code.namespace'])
    if span['namespace']:
        span['class_name'] = span['namespace'].split('.')[-1]
        span['package']    = '.'.join(span['namespace'].split('.')[:-1])
    else:
        span['class_name'] = ''
        span['package']    = ''

    span['code_function'] = get_attr_compiled(line, ATTR_PATTERNS['code.function'])

    span['http_route']   = get_attr_compiled(line, ATTR_PATTERNS['http.route'])
    span['db_statement'] = get_attr_compiled(line, ATTR_PATTERNS['db.statement'])

    span['url_path']     = get_attr_compiled(line, ATTR_PATTERNS['url.path'])
    span['url_query']    = get_attr_compiled(line, ATTR_PATTERNS['url.query'])
    span['status_code']  = get_attr_compiled(line, ATTR_PATTERNS['http.response.status_code'])

    db_table = get_attr_compiled(line, ATTR_PATTERNS['db.sql.table'])
    if not db_table:
        m = DB_TABLE_RE.search(line)
        if m:
            db_table = m.group(1).lower()
    span['db_table'] = db_table

    return span


from tqdm import tqdm
all_spans = [parse_line(l) for l in tqdm(RAW_LOG_LINES)]
df_raw    = pd.DataFrame(all_spans)
df        = df_raw[df_raw['namespace'].str.len() > 0].copy()

# ─── FILTER 1: AspectJ / JVM synthetic generated classes ─────────────────
SYNTHETIC_PATTERN = re.compile(
    r'(\$AjcClosure\d+|\$\$SpringCGLIB\$\$\d+|\$\$Lambda\$|\$Proxy\d+)',
    re.IGNORECASE
)

# ─── FILTER 2: Framework / server internal classes ────────────────────────
FRAMEWORK_PACKAGES = [
    'org.apache.catalina',
    'org.apache.tomcat',
    'org.apache.coyote',
    'org.eclipse.jetty',
    'jakarta.servlet',
    'javax.servlet',
    'org.springframework.web.servlet',
    'org.springframework.orm',
    'org.springframework.transaction',
    'org.springframework.aop',
    'org.springframework.cglib',
    'org.springframework.security.web',
    'org.hibernate.engine',
    'org.hibernate.event',
    'org.hibernate.loader',
    'org.hibernate.persister',
    'org.hibernate.proxy',
    'java.lang',
    'java.util',
    'sun.',
    'com.sun.',
]

def is_framework(namespace):
    ns = (namespace or '').lower()
    return any(ns.startswith(pkg.lower()) for pkg in FRAMEWORK_PACKAGES)

is_synthetic = df['class_name'].str.contains(SYNTHETIC_PATTERN, regex=True)
is_fw        = df['namespace'].apply(is_framework)
is_noise     = is_synthetic | is_fw

removed_synthetic = sorted(df[is_synthetic]['class_name'].unique())
removed_fw        = sorted(df[is_fw & ~is_synthetic]['class_name'].unique())
df                = df[~is_noise].copy()

print(f'Total log lines                  : {len(df_raw):,}')
print(f'Lines with a class name          : {len(df_raw[df_raw["namespace"].str.len()>0]):,}')
print()
print(f'Removed — AspectJ/JVM synthetic ({len(removed_synthetic)} classes):')
for r in removed_synthetic: print(f'  - {r}')
print()
print(f'Removed — Framework/server internals ({len(removed_fw)} classes):')
for r in removed_fw: print(f'  - {r}')
print()
print(f'Your application classes remaining: {df["class_name"].nunique():,}')
print(f'Unique request traces             : {df["trace_id"].nunique():,}')
print()
print('All classes that will be analysed:')
for cls in sorted(df['class_name'].unique()):
    pkg = df[df['class_name'] == cls]['package'].iloc[0]
    print(f'  {cls:<45}  [{pkg}]')

# ── Print db_table extraction
all_db_tables = sorted(df_raw[df_raw['db_table'].str.len() > 0]['db_table'].str.lower().unique())
print(f'\nDB tables detected in traces     : {len(all_db_tables)}')
for t in all_db_tables:
    print(f'  - {t}')

---
## Build the three input matrices


In [ ]:
from collections import Counter, defaultdict
import re

# ── Config ────────────────────────────────────────────────────────────────
INCLUDE_ERROR_TRACES = True
STATIC_ASSET_PATTERNS = (r'/webjars/', r'/css/', r'/js/', r'\.ico$',
                          r'\.png$', r'\.jpg$', r'\.svg$', r'\.map$')
ENTRYPOINT_SUFFIXES = ('ActionBean', 'Controller', 'Action', 'Resource',
                        'Servlet', 'Handler', 'Endpoint')

def is_static_asset(url_path):
    if not url_path:
        return False
    return any(re.search(pat, url_path) for pat in STATIC_ASSET_PATTERNS)

# classify SQL → operation label ─────────────────────────────────
def classify_sql_op(sql):
    if not sql:
        return ''
    op_match = re.match(r'\s*(SELECT|INSERT|UPDATE|DELETE|MERGE)', sql, re.IGNORECASE)
    if not op_match:
        return ''
    return {
        'SELECT': 'READ', 'INSERT': 'INSERT',
        'UPDATE': 'UPDATE', 'DELETE': 'DELETE', 'MERGE': 'MERGE',
    }.get(op_match.group(1).upper(), op_match.group(1).upper())

def classify_sql(sql):
    if not sql:
        return '', ''
    op = classify_sql_op(sql)
    if not op:
        return '', ''
    tbl_match = re.search(r'(?:FROM|INTO|UPDATE|JOIN)\s+([\w\.]+)', sql, re.IGNORECASE)
    table = tbl_match.group(1).lower().split('.')[-1] if tbl_match else ''
    return op, table

# ── Route derivation ───────────────────────────────────────
def is_glob_wildcard_route(route):

    if not route:
        return True
    return '*' in route

def derive_route_from_url(url_path, url_query):

    if not url_path:
        return None
    controller = url_path.rstrip('/').split('/')[-1]
    action = None
    if url_query:
        first_param = url_query.split('&')[0]
        action = first_param.split('=')[0].strip()
    return f"{controller}#{action}" if action else controller

def derive_route_from_entrypoint(trace_id, df_raw):

    trace_rows = df_raw[df_raw['trace_id'] == trace_id]
    internal = trace_rows[trace_rows['kind'] == 'INTERNAL']
    entry = internal[internal['class_name'].str.endswith(ENTRYPOINT_SUFFIXES, na=False)]
    candidates = entry if not entry.empty else internal
    for _, r in candidates.iterrows():
        if r.get('class_name') and r.get('code_function'):
            return f"{r['class_name']}.{r['code_function']}"
    return None

# ── Per-trace maps ────────────────────────────────────────────────────────
trace_groups = (
    df.groupby('trace_id')['class_name']
    .apply(lambda x: list(set(x)))
    .to_dict()
)

trace_to_route = {}
route_tier_counts = Counter()

for trace_id, grp in df_raw[df_raw['http_route'] != ''].groupby('trace_id'):
    row0 = grp.iloc[0]
    url_path = row0.get('url_path', '')

    if not INCLUDE_ERROR_TRACES:
        status = row0.get('status_code', '')
        if status and status.isdigit() and int(status) >= 400:
            continue

    if is_static_asset(url_path):
        continue  # drop webjars/css/js noise, not business logic

    route = row0.get('http_route', '')

    # Tier 1
    if not is_glob_wildcard_route(route):
        trace_to_route[trace_id] = route
        route_tier_counts['templated_route'] += 1
        continue

    # Tier 2
    fine = derive_route_from_url(url_path, row0.get('url_query', ''))
    if fine:
        trace_to_route[trace_id] = fine
        route_tier_counts['url_path_fallback'] += 1
        continue

    # Tier 3
    fine = derive_route_from_entrypoint(trace_id, df_raw)
    if fine:
        trace_to_route[trace_id] = fine
        route_tier_counts['entrypoint_fallback'] += 1
        continue


    trace_to_route[trace_id] = route
    route_tier_counts['raw_glob_unresolved'] += 1

print('Route derivation coverage:')
for tier, cnt in route_tier_counts.items():
    print(f'  {tier:<22}: {cnt} traces')
print()

# ── Build trace → table ────────
trace_to_table_ops = defaultdict(lambda: defaultdict(set))
for _, row in df_raw.iterrows():
    db_table = row.get('db_table', '').strip().lower()
    db_stmt  = row.get('db_statement', '')

    if db_table:
        table = db_table
        op    = classify_sql_op(db_stmt) if db_stmt else 'READ'
    elif db_stmt:
        op, table = classify_sql(db_stmt)
    else:
        continue

    if table and op:
        trace_to_table_ops[row['trace_id']][table].add(op)

all_project_classes = sorted(df['class_name'].unique())

# ═══════════════════════════════════════════════════════════════════════════
# MATRIX 1 — HTTP Route × Class  (weight = distinct trace count)
# ═══════════════════════════════════════════════════════════════════════════
route_class_counter = Counter()

for trace_id, classes in trace_groups.items():
    route = trace_to_route.get(trace_id)
    if not route:
        continue
    for cls in classes:
        route_class_counter[(route, cls)] += 1

all_routes = sorted({r for (r, _) in route_class_counter})

if all_routes:
    route_class_rows = []
    for route in all_routes:
        row = {'HTTP_Route': route}
        for cls in all_project_classes:
            row[cls] = route_class_counter.get((route, cls), 0)
        route_class_rows.append(row)
    route_class_df = pd.DataFrame(route_class_rows).set_index('HTTP_Route')
    route_class_df = route_class_df.loc[:, (route_class_df != 0).any(axis=0)]
    print('=== MATRIX 1 — HTTP Route × Class (trace counts) ===')
    display(route_class_df)
else:
    route_class_df = pd.DataFrame(index=pd.Index([], name='HTTP_Route'),
                                  columns=all_project_classes).fillna(0)
    print('No HTTP routes found in the trace log.')
print()



# ═══════════════════════════════════════════════════════════════════════════
# MATRIX 2 — Class co-occurrence  (weight = distinct trace count)
# ═══════════════════════════════════════════════════════════════════════════
cooc_counter = Counter()

for trace_id, classes in trace_groups.items():
    for c1, c2 in itertools.combinations(sorted(classes), 2):
        cooc_counter[(c1, c2)] += 1

cooc_rows = []
for c1 in all_project_classes:
    row = {'Class': c1}
    for c2 in all_project_classes:
        if c1 == c2:
            row[c2] = 0
        else:
            key = (min(c1, c2), max(c1, c2))
            row[c2] = cooc_counter.get(key, 0)
    cooc_rows.append(row)

cooc_df = pd.DataFrame(cooc_rows).set_index('Class')
print('=== MATRIX 2 — Class co-occurrence (distinct trace count) ===')
display(cooc_df)
print()

# ═══════════════════════════════════════════════════════════════════════════
# MATRIX 3 — Class × Table interaction  (weight = distinct trace count)
# ═══════════════════════════════════════════════════════════════════════════
class_table_op_counter = Counter()

for trace_id, classes in trace_groups.items():
    table_ops = trace_to_table_ops.get(trace_id, {})
    for cls in classes:
        for table, ops in table_ops.items():
            for op in ops:
                class_table_op_counter[(cls, table, op)] += 1


for _, row in df.iterrows():
    db_table = row.get('db_table', '').strip().lower()
    db_stmt  = row.get('db_statement', '')

    if db_table:
        op = classify_sql_op(db_stmt) if db_stmt else 'READ'
        table = db_table
    elif db_stmt:
        op, table = classify_sql(db_stmt)
    else:
        continue

    if table and op:
        class_table_op_counter[(row['class_name'], table, op)] += 1

all_tables  = sorted({tbl for (_, tbl, _) in class_table_op_counter})
all_ops     = ['READ', 'INSERT', 'UPDATE', 'DELETE', 'MERGE']

matrix3_rows = []
for cls in all_project_classes:
    row = {'Class': cls}
    for tbl in all_tables:
        for op in all_ops:
            val = class_table_op_counter.get((cls, tbl, op), 0)
            if val > 0:
                row[f'{tbl}.{op}'] = val
    matrix3_rows.append(row)

matrix_df = pd.DataFrame(matrix3_rows).fillna(0)
nonzero_cols = ['Class'] + [
    c for c in matrix_df.columns if c != 'Class' and matrix_df[c].sum() > 0
]
matrix_df = matrix_df[nonzero_cols]
for c in matrix_df.columns:
    if c != 'Class':
        matrix_df[c] = matrix_df[c].astype(int)

print('=== MATRIX 3 — Class × Table interaction (trace counts) ===')
print('Columns: table.OPERATION  |  Values: number of distinct traces')
display(matrix_df.set_index('Class'))
print()

# ── class summary
all_tables_in_traces = all_tables

class_info = []
for cls in all_project_classes:
    cls_rows  = df[df['class_name'] == cls]
    routes    = sorted(set(cls_rows[cls_rows['http_route'] != '']['http_route']))
    db_detail = defaultdict(dict)
    for (c, tbl, op), cnt in class_table_op_counter.items():
        if c == cls:
            db_detail[tbl][op] = cnt
    db_parts = []
    for tbl in sorted(db_detail.keys()):
        ops_str = ', '.join(f'{op}×{cnt}' for op, cnt in sorted(db_detail[tbl].items()))
        db_parts.append(f'{tbl}({ops_str})')
    class_info.append({
        'Class'          : cls,
        'Package'        : cls_rows['package'].mode()[0] if not cls_rows.empty else '',
        'HTTP_Routes'    : ', '.join(routes) if routes else '(none)',
        'DB_Interactions': '; '.join(db_parts) if db_parts else '(none)',
        'DB_Tables'      : ', '.join(sorted(db_detail.keys())) if db_detail else '(none)',
    })
info_df = pd.DataFrame(class_info)

print(f'Total project classes : {len(all_project_classes)}')
print(f'HTTP routes seen      : {len(all_routes)}')
print(f'DB tables seen        : {len(all_tables)}')

---
##  Build the prompt for the LLM


In [ ]:
def build_prompt(info_df, route_class_df, cooc_df, matrix_df):
    """
    Build the LLM prompt using three matrices:
      Matrix 1 — HTTP Route × Class    (trace-count weights)
      Matrix 2 — Class co-occurrence   (trace-count weights)
      Matrix 3 — Class × Table         (trace-count weights, split by operation)
    """

    # ── Matrix 1
    if not route_class_df.empty:
        cls_cols = list(route_class_df.columns)
        m1_lines = []
        header = f"{'HTTP_Route':<40}" + ''.join(f"{c:<30}" for c in cls_cols)
        m1_lines.append(header)
        m1_lines.append('-' * (40 + 30 * len(cls_cols)))
        for route, row in route_class_df.iterrows():
            line = f"{route:<40}"
            for c in cls_cols:
                val = int(row[c])
                line += f"{(str(val) if val > 0 else '.'):<30}"
            m1_lines.append(line)
        matrix1_text = '\n'.join(m1_lines)
        matrix1_block = f"=== MATRIX 1 — HTTP ROUTE × CLASS ===\nRows = HTTP routes. Columns = Java classes.\nValues = number of distinct traces where that class was active while serving that route.\nA dot (.) means zero.\nClasses with high weights on a route are the primary handlers of that route.\n\n{matrix1_text}\n"
    else:
        matrix1_block = "(Matrix 1 not available — no HTTP routes were found in the trace log. Use Matrix 2 and Matrix 3 only.)"

    # ── Matrix 2: Class co-occurrence ─────────────────────────────────────────
    cooc_classes = list(cooc_df.columns)
    m2_lines = []
    header = f"{'Class':<40}" + ''.join(f"{c:<25}" for c in cooc_classes)
    m2_lines.append(header)
    m2_lines.append('-' * (40 + 25 * len(cooc_classes)))
    for cls, row in cooc_df.iterrows():
        has_any = any(int(row[c]) > 0 for c in cooc_classes)
        if not has_any:
            continue
        line = f"{cls:<40}"
        for c in cooc_classes:
            val = int(row[c])
            line += f"{(str(val) if val > 0 else '.'):<25}"
        m2_lines.append(line)
    matrix2_text = '\n'.join(m2_lines)

    # ── Matrix 3: Class × Table interaction ───────────────────────────────────
    mat = matrix_df.set_index('Class')
    col_names = list(mat.columns)
    m3_lines = []
    header = f"{'Class':<40}" + ''.join(f"{c:<22}" for c in col_names)
    m3_lines.append(header)
    m3_lines.append('-' * (40 + 22 * len(col_names)))
    for cls, mat_row in mat.iterrows():
        non_zero = any(mat_row[c] > 0 for c in col_names)
        if not non_zero:
            continue
        line = f"{cls:<40}"
        for c in col_names:
            val = int(mat_row[c])
            line += f"{(str(val) if val > 0 else '.'):<22}"
        m3_lines.append(line)
    matrix3_text = '\n'.join(m3_lines)

    # ── Complete lists ─────────────────────────────────────────────────────────
    all_class_list = ', '.join(sorted(info_df['Class'].tolist()))
    all_table_list = ', '.join(sorted({tbl for (_, tbl, _) in class_table_op_counter}))

    prompt = f"""You are a software architect. Your task is to decompose a monolithic Java application
into microservices based on runtime behaviour captured from distributed execution traces.

You are given THREE matrices extracted from real runtime traces.
IMPORTANT: You must work ONLY with the classes and tables explicitly listed below.
Do NOT invent, assume, or infer any class or table not present in these matrices.

{matrix1_block}

=== MATRIX 2 — CLASS CO-OCCURRENCE ===
Rows and Columns = Java classes.
Values = number of distinct traces where BOTH classes appeared together in the same request.
A dot (.) means the two classes never appeared in the same trace.
High co-occurrence weight = strong runtime coupling → these classes likely belong to the same microservice.

{matrix2_text}

=== MATRIX 3 — CLASS × TABLE INTERACTION ===
Columns are "tablename.OPERATION" pairs.
Values = number of distinct traces where that class performed that operation on that table.
A dot (.) means zero.
Operations: READ=SELECT, INSERT, UPDATE, DELETE.
Write operations (INSERT/UPDATE/DELETE) signal table ownership.

{matrix3_text}

=== COMPLETE CLASS LIST (every class must appear in your answer exactly once) ===
{all_class_list}

=== COMPLETE TABLE LIST (every table must be owned by exactly one microservice) ===
{all_table_list}

=== GROUPING RULES ===

Rule 1 — HTTP Route prefix (strongest signal):
  Classes that appear together on the same URL prefix belong to the same microservice.
  Use Matrix 1: group classes that share the same route patterns (e.g. /orders/**, /catalog/**).

Rule 2 — Runtime co-occurrence (strong signal):
  Use Matrix 2: classes with high co-occurrence weights are tightly coupled at runtime.
  Prefer grouping highly co-occurring classes into the same microservice.
  Classes that never co-occur are candidates for different microservices.

Rule 3 — Table ownership via write operations:
  Use Matrix 3: a class that INSERTs, UPDATEs, or DELETEs a table OWNS that table.
  The owning class and its table belong to the same microservice.
  A class that only READs a table may be a consumer of another service's data.

Rule 4 — Completeness:
  EVERY class in the class list must be assigned to exactly one microservice.


=== OUTPUT FORMAT ===
Return ONLY a valid JSON object. No explanation. No markdown. No extra text.
Each microservice must list both the classes that implement it AND the tables it owns.
{{
  "MicroserviceName": {{
    "classes": ["ClassName1", "ClassName2"],
    "tables":  ["table_a", "table_b"]
  }},
  "AnotherMicroserviceName": {{
    "classes": ["ClassName3"],
    "tables":  ["table_c"]
  }}
}}
"""
    return prompt


PROMPT = build_prompt(info_df, route_class_df, cooc_df, matrix_df)

print(f'Prompt built: {len(PROMPT.split()):,} words')
print(f'Classes : {len(info_df)}')
print(f'Routes  : {len(route_class_df)}')
print(f'Tables  : {len(all_tables_in_traces)}')
print()
print('=== FULL PROMPT ===')
print(PROMPT)


---
## Send the prompt to both LLMs


In [ ]:
import json
import re
import time
import traceback
import requests
from collections import Counter
from itertools import combinations

SYSTEM_MESSAGE = (
    "You are an expert software architect specialising in microservice decomposition. "
    "You MUST identify multiple distinct microservices — never group everything into one service. "
    "The minimum number of microservices is 2 . "
    "Group classes by domain boundaries, shared data access, and functional cohesion. "
    "You always respond with valid JSON only. "
    "Do not use markdown fences. "
    "Return only a valid JSON object."
)


def call_llm(provider, model_id, prompt, system=SYSTEM_MESSAGE):


    if provider == "anthropic":
        url = ENDPOINTS["anthropic"]
        headers = {
            "Content-Type"     : "application/json",
            "x-api-key"        : ANTHROPIC_API_KEY,
            "anthropic-version": "2023-06-01",
        }
        payload = {
            "model"      : model_id,
            "max_tokens" : MAX_TOKENS,
            "system"     : system,
            "messages"   : [
                {"role": "user", "content": prompt},
            ],
        }

        # Opus 4.8 rejects a fixed temperature
        if "opus-4-8" not in model_id:
            payload["temperature"] = TEMPERATURE

        resp = requests.post(url, headers=headers, json=payload, timeout=90)

        if resp.status_code != 200:
            print("\n=== API ERROR RESPONSE ===")
            print(resp.text)
            print("==========================\n")

        resp.raise_for_status()
        data = resp.json()

        try:
            content = "".join(
                block.get("text", "")
                for block in data["content"]
                if isinstance(block, dict) and block.get("type") == "text"
            )
        except Exception:
            print("\n=== FULL RESPONSE ===")
            print(json.dumps(data, indent=2))
            print("=====================\n")
            raise

        return str(content).strip()

    # ── Provider setup (OpenAI-compatible providers: groq, gemini, openrouter, openai) ──
    if provider == "gemini":
        url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
        headers = {
            "Content-Type" : "application/json",
            "Authorization": f"Bearer {GEMINI_API_KEY}",
        }
    else:
        url = ENDPOINTS[provider]
        headers = {
            "Content-Type" : "application/json",
            "Authorization": f"Bearer {PROVIDER_KEYS[provider]}",
        }

    # ── Request payload ─────────────────────────────────────────────────────
    use_json_format = not (provider == "openrouter" and "gpt-oss" in model_id)

    payload = {
        "model"      : model_id,
        "messages"   : [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
    }

    # GPT-5
    # GPT-5 support the default temperature (1.0)
    if provider == "openai":
        payload["max_completion_tokens"] = MAX_TOKENS
    else:
        payload["max_tokens"]  = MAX_TOKENS
        payload["temperature"] = TEMPERATURE

    if use_json_format:
        payload["response_format"] = {"type": "json_object"}

    # ── API call ────────────────────────────────────────────────────────────
    resp = requests.post(url, headers=headers, json=payload, timeout=90)

    if resp.status_code != 200:
        print("\n=== API ERROR RESPONSE ===")
        print(resp.text)
        print("==========================\n")

    resp.raise_for_status()

    data = resp.json()

    try:
        content = data["choices"][0]["message"]["content"]
    except Exception:
        print("\n=== FULL RESPONSE ===")
        print(json.dumps(data, indent=2))
        print("=====================\n")
        raise

    if isinstance(content, list):
        content = " ".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict)
        )

    return str(content).strip()

"""Extract JSON object """
def parse_json_response(raw_text):


    if not raw_text or raw_text.strip() == "None":
        raise ValueError("Empty response from model.")

    text = re.sub(r"```(?:json)?\s*", "", raw_text, flags=re.IGNORECASE)
    text = text.replace("```", "").strip()

    start = text.find("{")
    end   = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in response:\n{text[:500]}")

    cleaned = text[start:end + 1]

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print("\n=== INVALID JSON ===")
        print(cleaned[:2000])
        print("====================\n")
        raise e


def normalise_decomp(decomp):


    normalised = {}

    for svc, value in decomp.items():

        if isinstance(value, list):
            classes = value
            owned_tables = sorted({
                tbl
                for cls in classes
                for (c, tbl, op) in class_table_op_counter
                if c == cls and op in ("INSERT", "UPDATE", "DELETE")
            })
            normalised[svc] = {"classes": classes, "tables": owned_tables}

        elif isinstance(value, dict):
            normalised[svc] = {
                "classes": value.get("classes", []),
                "tables" : value.get("tables",  []),
            }

        else:
            normalised[svc] = {"classes": [], "tables": []}

    return normalised


def fix_decomp(decomp):


    # ── Missing classes → UnassignedService ───────────────────────────────
    assigned_classes = set(c for v in decomp.values() for c in v["classes"])
    missing = all_expected_classes - assigned_classes

    if missing:
        decomp.setdefault("UnassignedService", {"classes": [], "tables": []})
        decomp["UnassignedService"]["classes"].extend(sorted(missing))

    # tables
    assigned_tables = set(t for v in decomp.values() for t in v["tables"])
    unowned_tables  = set(all_tables_in_traces) - assigned_tables

    for tbl in sorted(unowned_tables):
        best_svc, best_score = None, -1
        for svc, content in decomp.items():
            score = sum(
                class_table_op_counter.get((cls, tbl, op), 0) *
                (3 if op in ("INSERT", "UPDATE", "DELETE") else 1)
                for cls in content["classes"]
                for op in ("INSERT", "UPDATE", "DELETE", "READ")
            )
            if score > best_score:
                best_svc, best_score = svc, score
        if best_svc and tbl not in decomp[best_svc]["tables"]:
            decomp[best_svc]["tables"].append(tbl)

    return decomp

 """Jaccard similarity between two sets."""
def jaccard(set_a, set_b):

    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)

"""Return {class_name: service_name} mapping for a decomposition."""
def class_to_service_map(decomp):

    mapping = {}
    for svc, content in decomp.items():
        for cls in content["classes"]:
            mapping[cls] = svc
    return mapping


def jaccard_between_runs(decomp_a, decomp_b, all_classes):
    """
    Compute Jaccard similarity between two decompositions.
    For each class pair, check if they are in the same service in both runs.
    Jaccard = agreements / (agreements + disagreements)
    """
    map_a = class_to_service_map(decomp_a)
    map_b = class_to_service_map(decomp_b)

    classes  = sorted(all_classes)
    agree    = 0
    disagree = 0

    for cls1, cls2 in combinations(classes, 2):
        same_a = map_a.get(cls1) == map_a.get(cls2)
        same_b = map_b.get(cls1) == map_b.get(cls2)
        if same_a == same_b:
            agree += 1
        else:
            disagree += 1

    total = agree + disagree
    return agree / total if total > 0 else 1.0


def majority_vote(decomp_list, all_classes):
    """
    Produce a consensus decomposition by majority vote.
    Each class is assigned to the service it appeared in most across all runs.
    """
    # Count service assignments per class across runs
    class_votes = {cls: Counter() for cls in all_classes}

    for decomp in decomp_list:
        for svc, content in decomp.items():
            for cls in content["classes"]:
                if cls in class_votes:
                    class_votes[cls][svc] += 1

    # Build consensus decomposition
    consensus = {}
    for cls, votes in class_votes.items():
        if votes:
            best_svc = votes.most_common(1)[0][0]
            consensus.setdefault(best_svc, {"classes": [], "tables": []})
            consensus[best_svc]["classes"].append(cls)

    # assign only tables owned by each service's classes
    for svc, content in consensus.items():
        owned_tables = sorted({
            tbl
            for cls in content["classes"]
            for (c, tbl, op) in class_table_op_counter
            if c == cls and op in ("INSERT", "UPDATE", "DELETE")
        })
        content["tables"] = owned_tables   # ← fixes all-tables-in-every-service bug

    return fix_decomp(consensus)


# ── Ground truth ────────────────────────────────────────────────────────────
all_tables_in_traces  = sorted({tbl for (_, tbl, _) in class_table_op_counter})
all_expected_classes  = set(info_df["Class"].tolist())

# ── Storage ──────────────────────────────────────────────────────────────────
results            = {}   # final (majority vote) decomposition per model
all_runs           = {}   # all N_RUNS raw decompositions per model
consistency_scores = {}   # avg Jaccard per model

# ── Run all models N_RUNS times ──────────────────────────────────────────────
for label, (provider, model_id) in MODELS.items():

    print(f"\n{'=' * 80}")
    print(f"Model : {label} ({model_id})")
    print(f"Runs  : {N_RUNS}")
    print(f"{'=' * 80}")

    run_decomps = []

    for run_idx in range(1, N_RUNS + 1):

        print(f"\n  ── Run {run_idx}/{N_RUNS} ──")

        try:
            raw    = call_llm(provider, model_id, PROMPT)
            print(f"  RAW (first 300 chars): {raw[:300]}")
            decomp = fix_decomp(normalise_decomp(parse_json_response(raw)))
            run_decomps.append(decomp)

            assigned_classes = set(c for v in decomp.values() for c in v["classes"])
            coverage = len(assigned_classes & all_expected_classes) / len(all_expected_classes) * 100
            print(f"  Services : {len(decomp)}  |  Coverage : {coverage:.0f}%")

        except Exception as e:
            print(f"  ERROR on run {run_idx}: {e}")
            traceback.print_exc()


        sleep_time = 8 if provider == "openrouter" else 3
        time.sleep(sleep_time)

    if not run_decomps:
        print(f"\n  All runs failed — skipping {label}")
        results[label]            = {}
        consistency_scores[label] = 0.0
        all_runs[label]           = []
        continue

    # ── Jaccard consistency across runs ─────────────────────────────────────
    if len(run_decomps) > 1:
        pairwise = [
            jaccard_between_runs(a, b, all_expected_classes)
            for a, b in combinations(run_decomps, 2)
        ]
        avg_jaccard = sum(pairwise) / len(pairwise)
    else:
        pairwise    = [1.0]
        avg_jaccard = 1.0

    consistency_scores[label] = avg_jaccard
    all_runs[label]           = run_decomps

    print(f"\n  ── Consistency ──")
    for i, (a, b) in enumerate(combinations(range(1, len(run_decomps) + 1), 2)):
        print(f"  Jaccard (Run{a} vs Run{b}) : {pairwise[i]:.3f}")
    print(f"  Avg Jaccard            : {avg_jaccard:.3f}  ", end="")
    if avg_jaccard >= 0.80:
        print("Stable")
    elif avg_jaccard >= 0.60:
        print("Moderate")
    else:
        print("Unstable")

    # ── Majority vote final decomposition ───────────────────────────────────
    final_decomp   = majority_vote(run_decomps, all_expected_classes)
    results[label] = final_decomp

    print(f"\n  ── Final (Majority Vote) ──")
    print(f"  Microservices : {len(final_decomp)}")
    for svc, content in final_decomp.items():
        print(f"\n    {svc}")
        print(f"      classes : {content['classes']}")
        print(f"      tables  : {content['tables']}")

# ── Consistency summary ──────────────────────────────────────────────────────
print(f"\n\n{'=' * 80}")
print("CONSISTENCY SUMMARY")
print(f"{'=' * 80}")
print(f"{'Model':<25} {'Avg Jaccard':>12} {'Stability':>12}")
print("-" * 50)
for label, score in consistency_scores.items():
    stability = "Stable" if score >= 0.80 else ("Moderate" if score >= 0.60 else "Unstable")
    print(f"{label:<25} {score:>12.3f} {stability:>12}")

---
## Visualise the decomposition graphs


In [ ]:
# ── Build the co-occurrence graph ─────────────────────────────────────────────
G = nx.Graph()
edge_counts = defaultdict(int)
for trace_id, classes in trace_groups.items():
    for c1, c2 in itertools.combinations(sorted(classes), 2):
        edge_counts[(c1, c2)] += 1
for (c1, c2), w in edge_counts.items():
    G.add_edge(c1, c2, weight=w)
for cls in info_df['Class']:
    G.add_node(cls)

PALETTE = list(mcolors.TABLEAU_COLORS.values())

def draw_decomposition(G, decomp, title, ax):
    """Draw co-occurrence graph coloured by microservice.
    decomp shape: {service: {"classes": [...], "tables": [...]}}
    """
    services     = list(decomp.keys())
    svc_to_color = {svc: PALETTE[i % len(PALETTE)] for i, svc in enumerate(services)}

    # Map each node (class) to its service
    node_to_svc = {
        cls: svc
        for svc, content in decomp.items()
        for cls in content["classes"]
    }
    node_colors = [svc_to_color.get(node_to_svc.get(n, ''), '#dddddd') for n in G.nodes()]

    # Edge = co-occurrence frequency
    weights = [G[u][v].get('weight', 1) for u, v in G.edges()]
    max_w   = max(weights) if weights else 1
    widths  = [2.5 * w / max_w for w in weights]

    pos = nx.spring_layout(G, seed=42, k=1.8)
    nx.draw_networkx_nodes(G,  pos, node_color=node_colors, node_size=1000, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
    nx.draw_networkx_edges(G,  pos, width=widths, alpha=0.45,
                           edge_color='#666666', ax=ax)

    # Legend: service name + owned tables
    patches = []
    for svc in services:
        tables = decomp[svc].get("tables", [])
        tbl_str = ', '.join(tables) if tables else '—'
        label = f'{svc}\n  tables: {tbl_str}'
        patches.append(mpatches.Patch(color=svc_to_color[svc], label=label))
    ax.legend(handles=patches, loc='upper left', fontsize=6,
              title='Microservice (owned tables)', framealpha=0.9)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=10)
    ax.axis('off')


valid_results = {k: v for k, v in results.items() if v}
n = len(valid_results)

if n == 0:
    print('No valid results to visualise.')
else:
    fig, axes = plt.subplots(1, n, figsize=(9 * n, 7))
    if n == 1:
        axes = [axes]

    for ax, (label, decomp) in zip(axes, valid_results.items()):
        draw_decomposition(G, decomp, label, ax)

    plt.suptitle('Microservice Decomposition — Co-occurrence Graph',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('decomposition_graph.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Graph saved as decomposition_graph.png')




## Tradional Clustering




In [ ]:
import numpy as np

W_COOC  = 0.5   # Matrix 2 — behavioural coupling (already class×class)
W_ROUTE = 0.25  # Matrix 1 — classes that serve the same HTTP routes
W_TABLE = 0.25  # Matrix 3 — classes that access the same DB tables

classes = all_project_classes
n = len(classes)

S_cooc = cooc_df.reindex(index=classes, columns=classes, fill_value=0).to_numpy(dtype=float)

route_bin = (route_class_df.reindex(columns=classes, fill_value=0) > 0).to_numpy(dtype=float)
S_route = route_bin.T @ route_bin
np.fill_diagonal(S_route, 0)

m3 = matrix_df.set_index('Class').reindex(classes, fill_value=0)
table_names = sorted({c.rsplit('.', 1)[0] for c in m3.columns})
table_bin = np.zeros((n, len(table_names)))
for j, tbl in enumerate(table_names):
    cols = [c for c in m3.columns if c.rsplit('.', 1)[0] == tbl]
    table_bin[:, j] = (m3[cols].sum(axis=1) > 0).astype(float)
S_table = table_bin @ table_bin.T
np.fill_diagonal(S_table, 0)

def norm(M):
    mx = M.max()
    return M / mx if mx > 0 else M

S_combined = W_COOC * norm(S_cooc) + W_ROUTE * norm(S_route) + W_TABLE * norm(S_table)

print(f'Combined similarity matrix built for {n} classes.')
print(f'  Matrix 2 (co-occurrence) max weight : {S_cooc.max():.0f}')
print(f'  Matrix 1 (shared routes) max weight : {S_route.max():.0f}')
print(f'  Matrix 3 (shared tables) max weight : {S_table.max():.0f}')

# ── Print the combined matrix
S_combined_df = pd.DataFrame(S_combined, index=classes, columns=classes).round(3)

print(f'\nCombined similarity matrix ({n} × {n} classes):\n')
display(S_combined_df)


S_combined_df.to_csv('combined_similarity_matrix.csv')
print("\nSaved: combined_similarity_matrix.csv")

# Louvain Algorithm


In [ ]:
G_combined = nx.Graph()
G_combined.add_nodes_from(classes)
for i in range(n):
    for j in range(i + 1, n):
        w = S_combined[i, j]
        if w > 0:
            G_combined.add_edge(classes[i], classes[j], weight=w)

louvain_communities = nx.algorithms.community.louvain_communities(
    G_combined, weight='weight', seed=42, resolution=1.0
)

def clusters_to_decomp(clusters, matrix_df, prefix):
    """Turn a list of class-sets into {service: {classes, tables}}.
    A table goes to whichever cluster has the highest total read+write"""
    m3 = matrix_df.set_index('Class')
    table_totals = {tbl: {} for tbl in sorted({c.rsplit('.', 1)[0] for c in m3.columns})}
    for k, members in enumerate(clusters):
        sub = m3.reindex(list(members), fill_value=0)
        for col in sub.columns:
            tbl = col.rsplit('.', 1)[0]
            table_totals[tbl][k] = table_totals[tbl].get(k, 0) + sub[col].sum()

    table_owner = {
        tbl: max(counts, key=counts.get)
        for tbl, counts in table_totals.items() if counts and max(counts.values()) > 0
    }

    decomp = {}
    for k, members in enumerate(clusters):
        svc = f'{prefix}_{k+1}'
        decomp[svc] = {
            'classes': sorted(members),
            'tables' : sorted([t for t, owner in table_owner.items() if owner == k]),
        }
    return decomp

results = results if 'results' in dir() else {}

louvain_decomp = clusters_to_decomp(louvain_communities, matrix_df, 'LouvainService')
results['Louvain (3-Matrix Graph)'] = louvain_decomp

print(f'\n=== Louvain — {len(louvain_decomp)} microservices ===')
for svc, content in louvain_decomp.items():
    print(f'\n  {svc}')
    print(f'    classes : {content["classes"]}')
    print(f'    tables  : {content["tables"]}')

# Hierarchical Agg Clustering

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering


def clusters_to_decomp(clusters, matrix_df, prefix):
    """Turn a list of class-sets into {service: {classes, tables}}.
    A table goes to whichever cluster has the highest total read+write"""
    m3 = matrix_df.set_index('Class')
    table_totals = {tbl: {} for tbl in sorted({c.rsplit('.', 1)[0] for c in m3.columns})}
    for k, members in enumerate(clusters):
        sub = m3.reindex(list(members), fill_value=0)
        for col in sub.columns:
            tbl = col.rsplit('.', 1)[0]
            table_totals[tbl][k] = table_totals[tbl].get(k, 0) + sub[col].sum()

    table_owner = {
        tbl: max(counts, key=counts.get)
        for tbl, counts in table_totals.items() if counts and max(counts.values()) > 0
    }

    decomp = {}
    for k, members in enumerate(clusters):
        svc = f'{prefix}_{k+1}'
        decomp[svc] = {
            'classes': sorted(members),
            'tables' : sorted([t for t, owner in table_owner.items() if owner == k]),
        }
    return decomp


N_CLUSTERS = 5  # number of microservices in the reference decomposition

D = S_combined.max() - S_combined      # similarity distance
np.fill_diagonal(D, 0)
D = (D + D.T) / 2                      # enforce exact symmetry
condensed = squareform(D, checks=False)

Z = linkage(condensed, method='average')   #

plt.figure(figsize=(12, 5))
dendrogram(Z, labels=classes, leaf_rotation=90, leaf_font_size=8)
plt.title('Hierarchical Clustering — combined Matrix 1+2+3 similarity')
plt.ylabel('Distance')
plt.tight_layout()
plt.savefig('hierarchical_dendrogram.png', dpi=150)
plt.show()


agg = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric='precomputed', linkage='average')
cluster_labels = agg.fit_predict(D)

hier_clusters = defaultdict(list)
for cls, lbl in zip(classes, cluster_labels):
    hier_clusters[lbl].append(cls)
hier_clusters = list(hier_clusters.values())

hier_decomp = clusters_to_decomp(hier_clusters, matrix_df, 'HierService')
results['Hierarchical (3-Matrix Features)'] = hier_decomp

print(f'\n=== Hierarchical Clustering (k={N_CLUSTERS}) — {len(hier_decomp)} microservices ===')
for svc, content in hier_decomp.items():
    print(f'\n  {svc}')
    print(f'    classes : {content["classes"]}')
    print(f'    tables  : {content["tables"]}')

---
##  Export results


In [ ]:
import datetime
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Save class info table
info_df.to_csv(f'class_info_table_{ts}.csv', index=False)
print(f'Saved: class_info_table_{ts}.csv')

# Save each model's decomposition (includes classes + tables)
for label, decomp in results.items():
    safe_label = label.replace(' ', '_').replace('.', '')
    fname = f'decomposition_{safe_label}_{ts}.json'
    with open(fname, 'w') as f:
        json.dump(decomp, f, indent=2)
    print(f'Saved: {fname}')



# Download all files
try:
    from google.colab import files as _f
    for fname in [
        f'class_info_table_{ts}.csv',
        f'comparison_table_{ts}.csv',
        f'table_ownership_comparison_{ts}.csv',
        'decomposition_graph.png',
    ]:
        try:
            _f.download(fname)
        except Exception:
            pass
    for label in results:
        safe_label = label.replace(' ', '_').replace('.', '')
        try:
            _f.download(f'decomposition_{safe_label}_{ts}.json')
        except Exception:
            pass
    print('Download dialogs triggered.')
except ImportError:
    print('(Not in Colab — files saved to current directory.)')
